## Step 1 of the migration: snapshot the current workspace into this bundle

Run this notebook **once, in the old (going-away) workspace**. It copies the latest hackathon work into this bundle folder so it can be committed and pushed with the `2026-HACKATHON` Git repo:

| What | Copied from | Copied to (bundle) |
| --- | --- | --- |
| App code (release 20261102, served by app 2026h) | `hackathon_releases/20261102` | `src/app/` |
| Judge walkthrough notebook, README, frozen Oct 29 zip | `hackathon_submissions/20261029` | `src/judge/` |
| PA data load notebook | `Load FEMA Public Assistance Funded Projects` | `src/notebooks/` |
| Dashboard | FEMA Public Assistance Funding Overview | `src/dashboards/` |
| Genie space definition | FEMA Public Assistance Funding | `migration_data/genie_space.json` |
| Golden eval dataset (116 cases) | `workspace.default.fema_oversight_golden_v1` | `migration_data/golden_dataset.json` |
| MLflow run history (params, metrics, tags) | experiment 1305734463288268 | `migration_data/mlflow_runs.json` |
| Latest submission packet files | `/Volumes/users/phebe_arouni/hackathon_submissions/20261102` (top level only) | `migration_data/volume_files/20261102/` |

The last cell re-reads every copied file and checks its SHA-256 against the source. **Do not commit until it prints `ALL COPIES VERIFIED`.**

Not copied (by design): old release 20261029, the `fema-oversight-mcp` folder, superseded zips, and the rows of `funded_projects_details` (the restore job reloads them from the public OpenFEMA API).

In [0]:
import base64, hashlib, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

w = WorkspaceClient()
HOME = "/Workspace/Users/phebe.arouni@hq.dhs.gov"
BUNDLE = f"{HOME}/2026-HACKATHON/HACKATHON_BACKUP_2026_11_03"

RELEASE_DIR = f"{HOME}/hackathon_releases/20261102"
JUDGE_DIR = f"{HOME}/hackathon_submissions/20261029"
JUDGE_NOTEBOOK = f"{JUDGE_DIR}/00_JUDGES_START_HERE - FEMA Evidence and Oversight Graph Walkthrough"
JUDGE_FILES = ["README_FOR_JUDGES.md", "FEMA_Hackathon_Submission_2026-10-29.zip"]
LOAD_NOTEBOOK = f"{HOME}/Load FEMA Public Assistance Funded Projects"
DASHBOARD_ID = "01f1d14d0ccc16679f6602bde17cae23"
GENIE_SPACE_ID = "01f1cadd51e817f48393d89f8c01ac45"
GOLDEN_TABLE = "workspace.default.fema_oversight_golden_v1"
EXPERIMENT_ID = "1305734463288268"
VOLUME_DIR = "/Volumes/users/phebe_arouni/hackathon_submissions/20261102"

manifest = []  # one entry per copied item: source, target, sha256

In [0]:
def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def ws(path: str) -> str:
    """Workspace API path (no /Workspace prefix)."""
    return path[len("/Workspace"):] if path.startswith("/Workspace") else path

def _guard(target: str):
    # Only ever write inside the bundle; never overwrite anything (re-runs compare instead).
    assert target.startswith(BUNDLE + "/"), f"Refusing to write outside the bundle: {target}"

def _exists(target: str) -> bool:
    try:
        w.workspace.get_status(ws(target))
        return True
    except Exception:
        return False

def write_file(target: str, data: bytes):
    _guard(target)
    if os.path.exists(target):
        with open(target, "rb") as f:
            assert sha256(f.read()) == sha256(data), f"{target} already exists with different content - not overwriting"
        return
    w.workspace.mkdirs(ws(os.path.dirname(target)))
    w.workspace.import_(path=ws(target), content=base64.b64encode(data).decode(),
                        format=ImportFormat.RAW, overwrite=False)

def copy_file(source: str, target: str):
    with open(source, "rb") as f:
        data = f.read()
    write_file(target, data)
    manifest.append({"kind": "file", "source": source, "target": target, "sha256": sha256(data), "bytes": len(data)})

def copy_notebook(source: str, target_dir: str):
    """Copy a notebook as .ipynb (outputs included) so bundles can deploy it."""
    exp = w.workspace.export(ws(source), format=ExportFormat.JUPYTER)
    data = base64.b64decode(exp.content)
    target = f"{target_dir}/{os.path.basename(source)}"
    _guard(target)
    if not _exists(target):
        w.workspace.mkdirs(ws(target_dir))
        w.workspace.import_(path=ws(target), content=exp.content, format=ImportFormat.JUPYTER,
                            language=Language.PYTHON, overwrite=False)
    manifest.append({"kind": "notebook", "source": source, "target": target, "sha256": sha256(data), "bytes": len(data)})

def write_json(target: str, obj, source: str):
    data = json.dumps(obj, indent=2, default=str).encode()
    write_file(target, data)
    manifest.append({"kind": "export", "source": source, "target": target, "sha256": sha256(data), "bytes": len(data)})

In [0]:
n = 0
for root, dirs, files in os.walk(RELEASE_DIR):
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".pytest_cache", ".ruff_cache", ".venv")]
    for name in files:
        src = os.path.join(root, name)
        copy_file(src, f"{BUNDLE}/src/app/{os.path.relpath(src, RELEASE_DIR)}")
        n += 1
print(f"App files copied: {n} (expected 22: 19 .py + analyst.html + app.yaml + requirements.txt)")
assert n == 22, "Unexpected file count - check the release folder"

In [0]:
copy_notebook(JUDGE_NOTEBOOK, f"{BUNDLE}/src/judge")
for name in JUDGE_FILES:
    copy_file(f"{JUDGE_DIR}/{name}", f"{BUNDLE}/src/judge/{name}")
copy_notebook(LOAD_NOTEBOOK, f"{BUNDLE}/src/notebooks")
print("Judge package + load notebook copied")

In [0]:
dash = w.lakeview.get(DASHBOARD_ID)
serialized = json.loads(dash.serialized_dashboard)
data = json.dumps(serialized, indent=2).encode()
target = f"{BUNDLE}/src/dashboards/fema_pa_funding_overview.lvdash.json"
write_file(target, data)
manifest.append({"kind": "export", "source": f"dashboard {DASHBOARD_ID}", "target": target, "sha256": sha256(data), "bytes": len(data)})
print("Dashboard:", dash.display_name, "| datasets:", len(serialized.get("datasets", [])))

genie = w.api_client.do("GET", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}",
                        query={"include_serialized_space": "true"})
assert genie.get("serialized_space"), "Genie export returned no serialized_space"
write_json(f"{BUNDLE}/migration_data/genie_space.json",
           {"title": genie.get("title"), "description": genie.get("description"),
            "serialized_space": genie["serialized_space"]}, f"genie space {GENIE_SPACE_ID}")
print("Genie space:", genie.get("title"))

In [0]:
rows = spark.table(GOLDEN_TABLE).select("dataset_record_id", "inputs", "expectations", "tags").collect()
records = [{"dataset_record_id": r.dataset_record_id,
            "inputs": json.loads(r.inputs) if r.inputs else None,
            "expectations": json.loads(r.expectations) if r.expectations else None,
            "tags": dict(r.tags) if r.tags else {}} for r in rows]
write_json(f"{BUNDLE}/migration_data/golden_dataset.json", records, GOLDEN_TABLE)
print("Golden records:", len(records), "(expected 116)")

import mlflow
mlflow.set_tracking_uri("databricks")
runs = mlflow.search_runs(experiment_ids=[EXPERIMENT_ID], output_format="list", max_results=5000)
exported = [{"run_id": r.info.run_id, "run_name": r.info.run_name, "status": r.info.status,
             "start_time": r.info.start_time, "end_time": r.info.end_time,
             "params": r.data.params, "metrics": r.data.metrics,
             "tags": {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.") or k in ("mlflow.runName", "mlflow.note.content")}}
            for r in runs]
write_json(f"{BUNDLE}/migration_data/mlflow_runs.json", exported, f"experiment {EXPERIMENT_ID}")
print("MLflow runs exported:", len(exported))

In [0]:
n = 0
for entry in sorted(os.listdir(VOLUME_DIR)):
    src = f"{VOLUME_DIR}/{entry}"
    if os.path.isfile(src):  # top level only: skips superseded_rev1 / superseded_rev2
        copy_file(src, f"{BUNDLE}/migration_data/volume_files/20261102/{entry}")
        n += 1
print("Volume files copied:", n)

In [0]:
def read_back(item) -> bytes:
    if item["kind"] == "notebook":
        return base64.b64decode(w.workspace.export(ws(item["target"]), format=ExportFormat.JUPYTER).content)
    with open(item["target"], "rb") as f:
        return f.read()

def cell_sources(ipynb: bytes):
    nb = json.loads(ipynb)
    return [(c["cell_type"], "".join(c["source"]) if isinstance(c["source"], list) else c["source"])
            for c in nb["cells"]]

problems, metadata_only = [], []
for item in manifest:
    got_bytes = read_back(item)
    got = sha256(got_bytes)
    if got == item["sha256"]:
        continue
    if item["kind"] == "notebook":
        # Byte hash differs: fall back to comparing cell types + sources with a fresh source export
        src = base64.b64decode(w.workspace.export(ws(item["source"]), format=ExportFormat.JUPYTER).content)
        if cell_sources(src) == cell_sources(got_bytes):
            metadata_only.append(item["target"])
            continue
    problems.append((item["target"], item["sha256"][:12], got[:12]))

for t in metadata_only:
    print("OK (cell sources identical; export metadata differs):", t)

write_file(f"{BUNDLE}/migration_data/MANIFEST.json", json.dumps(manifest, indent=2).encode())
print(f"{len(manifest)} items copied")
for p in problems:
    print("MISMATCH", *p)
print("ALL COPIES VERIFIED" if not problems else f"{len(problems)} MISMATCHES - do not commit")
# Note: notebooks are compared as re-exported .ipynb; minor metadata differences from the
# workspace export can show as a mismatch. Compare cell sources if that happens.